# Confluent Schema Registry{background-image="https://docs.confluent.io/platform/current/_images/schema-registry-ecosystem.jpg" background-size="contain" background-repeat="no-repeat" background-position="right top"}
A shared contract for Kafka records


## Data contracts{background-image="https://docs.datacontract.com/assets/images/datacontractcli-7fd19880c35babb06d925e43ffb7a612.webp" background-size="contain" background-repeat="no-repeat" background-opacity="0.4" background-color="white" background-position="right top"}

:::{.fragment style="background-color: #f9f9f9; border-left: 4px solid #007acc; padding: 1em; background-opacity: 0.8;"}
The [Open Data Contract Standard (ODCS) v3.1.0](https://docs.datacontract.com/open-data-contract-standard) describes a data contract as a platform-agnostic YAML document that makes an agreement about a data product explicit. Its sections cover more than the payload schema:

- **Fundamentals:** identity, version, status, domain, purpose, usage, and limitations;
- **Schema:** objects and properties, with business meaning and physical representation;
- **Data quality:** rules and expectations for the data;
- **People and operations:** team and roles, support/communication channels, SLAs, and infrastructure.
:::

## Example: a group chat

A group chat can publish messages as events to the Kafka topic `tap-messages`. Each record describes one message:

```json
{
  "ts": 1792348935000,
  "id": "1",
  "message": "TAP lesson starts at 15"
}
```

Where  `ts` is the event time, `id` identifies this message, and `message` is its text. 

The producer and every consumer must agree on these fields and their meanings.

:::{.callout-note .fragment} 
If a consumer expects `message`, what happens if a producer silently renames it to `text`?
:::

## Shema Registry for data contract

- Schema registry is just a part of a data contract.
- The role is to help to manage the **machine-readable schema** part of a data contract: field structure, data types, schema versions, and configured compatibility checks.
- We will see other technologies more generally used for Data Governance including data contract management, for example, OpenMetadata

## Components: Format, schema, registry

- A **serialization format** represents data as bytes: for example Avro, Protobuf, or JSON.
- A **schema** describes the expected structure of data in a chosen format.
- A **Schema Registry** stores and versions schemas, and can check proposed changes against compatibility rules.
- A **serializer/deserializer** turns application data into bytes and back, using the agreed format and schema.


## Confluent Schema Registry

**Confluent Schema Registry** [`confluentinc/schema-registry`](https://github.com/confluentinc/schema-registry)

- It was developed in the Confluent/Kafka ecosystem and provides a REST API and client integrations.
- It supports Avro, Protobuf, and JSON Schema.
- It is not an Apache Software Foundation project, and the project is not uniformly Apache-2.0 licensed: the server uses the Confluent Community License


## Role of the Schema Registry in the Kafka ecosystem

```{mermaid}
%%| echo: false
sequenceDiagram
    participant P as Producer application
    participant S as Serializer
    participant SR as Schema Registry
    participant K as Kafka topic
    participant D as Deserializer
    participant C as Consumer application
    P->>S: record and schema
    S->>SR: register or look up schema for subject
    SR-->>S: schema ID
    S->>K: serialized record with schema ID
    K->>D: record bytes and schema ID
    D->>SR: fetch schema by ID (if not cached)
    SR-->>D: schema definition
    D-->>C: deserialized record
```

:::{.callout-note .fragment}
The registry is a separate service, not a Kafka broker and not the path through which every event is delivered. Kafka still carries the records. In the Confluent wire format, the serialized record includes a schema ID that lets a reader retrieve the corresponding schema.
:::

## Lifecycle of a record using schema registry

:::: {.columns}
::: {.column width="33%"}
**Actors**

- **Topic**: Kafka topic, e.g. `tap-messages`, maintains records 
- **Subject**: Schema Registry name that define schema's version history.
- **Schema ID**: Identifier of the the registered schema (global)

:::{.callout-note .fragment}
Default Naming Strategy: subject name is `<topic>-key` or `<topic>-value` depending on the record type.
:::
:::

::: {.column width="33%"}
**Schema formats**

- **Avro:** compact binary records with an explicit schema.
- **Protobuf:** message definitions use stable field numbers.
- **JSON Schema:** describes constraints on JSON-shaped data.

:::

::: {.column width="34%"}
**Lifecycle**

1. The producer serializer gets a schema.
2. The schema is registered or found under a subject.
3. The serializer writes the record with its schema ID.
4. The consumer deserializer uses that ID to interpret the record.

:::{.callout-note .fragment}
The payload stays in Kafka; the Registry stores schema metadata.
:::
:::

::::

## What does "compatible" mean?

Imagine that a producer has started writing records with a new schema, while some consumers may still use the old schema.

Compatibility asks whether readers using one schema can correctly interpret data written with another. It is a check on schema evolution, not a promise that the application's business meaning has stayed the same.

**Ask the class:** Should we check the new schema before or after deploying the producer?

## Three compatibility directions

| Policy | Reader | Data being read | In plain language |
|---|---|---|---|
| **Backward** | New schema | Old-schema data | Can new code read records already written? |
| **Forward** | Old schema | New-schema data | Can old code read records newly written? |
| **Full** | Both | Both directions | Can old and new versions interoperate? |

In Confluent Schema Registry, the default compatibility level is `BACKWARD`. The configured level can be changed globally or for an individual subject.

## Version 1: the chat-message event

Treat each published record as an immutable event: one message was sent at a particular time. The initial Avro schema is:

```json
{
  "type": "record",
  "name": "ChatMessage",
  "fields": [
    {"name": "ts", "type": {"type": "long", "logicalType": "timestamp-millis"}},
    {"name": "id", "type": "string"},
    {"name": "message", "type": "string"}
  ]
}
```

The timestamp is event time, not a guarantee about when Kafka received the record. `id` is the message/event identifier; the record key is a separate Kafka field and choice.

## Version 2: add sender and attachments

The chat now needs to identify the sender and describe optional attachments. Add both fields as nullable unions with `null` defaults so a new reader can still read version 1 records:

```json
{
  "type": "record",
  "name": "ChatMessage",
  "fields": [
    {"name": "ts", "type": {"type": "long", "logicalType": "timestamp-millis"}},
    {"name": "id", "type": "string"},
    {"name": "message", "type": "string"},
    {"name": "sender", "type": ["null", {"type": "record", "name": "Sender", "fields": [
      {"name": "id", "type": "string"},
      {"name": "displayName", "type": ["null", "string"], "default": null}
    ]}], "default": null},
    {"name": "attachments", "type": ["null", {"type": "array", "items": {"type": "record", "name": "Attachment", "fields": [
      {"name": "mediaType", "type": "string"},
      {"name": "fileName", "type": ["null", "string"], "default": null},
      {"name": "sizeBytes", "type": ["null", "long"], "default": null}
    ]}}], "default": null}
  ]
}
```

An old event has neither `sender` nor `attachments`; the defaults let the new Avro reader supply `null`. An event with no known attachment can use `null`; if the contract distinguishes “known to have none” from “not supplied,” agree whether an empty array `[]` has a different meaning.

## Not every change is safe

- Renaming a field can look like removing one field and adding another.
- Changing a field's type can make old data unreadable.
- Adding a required field may leave older records without a value.
- Keeping the same type while changing what a field *means* may pass a structural check and still break an application.

Compatibility is format-specific. Treat it as an important automated guardrail, then review the data's meaning and deployment plan with the teams that produce and consume it.

## Instructor demonstration: the Schema Registry API

The Confluent quickstart demonstrates the Registry through its REST API. We will adapt that sequence to the `group-chat-events` stream.

First start a single-node Kafka broker and Schema Registry with Docker. The API requests then register contracts for our chat-event topic; they still do not produce or consume chat records.

Run the commands one at a time in a terminal (for example, Bash or Git Bash). On Windows PowerShell, use `curl.exe` if `curl` resolves to a PowerShell alias.

**Ask before continuing:** Which service receives the HTTP request: Kafka or Schema Registry?

## Step 0: prepare the Docker network and Kafka

Use the same Kafka image and single-node KRaft settings as the Kafka lesson. Run these commands in Bash or Git Bash. Create the shared network only if it does not already exist:

```bash
docker network inspect tap >/dev/null 2>&1 || docker network create tap
```

If `kafkaServer` from the Kafka lesson is already running on `tap`, keep using it and skip `docker run`. Otherwise start the broker:

```bash
docker run --rm -d --name kafkaServer --network tap -p 9092:9092 \
  -e KAFKA_NODE_ID=1 \
  -e KAFKA_PROCESS_ROLES=broker,controller \
  -e KAFKA_LISTENERS=PLAINTEXT://:9092,CONTROLLER://:9093 \
  -e KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://kafkaServer:9092 \
  -e KAFKA_CONTROLLER_LISTENER_NAMES=CONTROLLER \
  -e KAFKA_LISTENER_SECURITY_PROTOCOL_MAP=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT \
  -e KAFKA_CONTROLLER_QUORUM_VOTERS=1@localhost:9093 \
  -e KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR=1 \
  -e KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR=1 \
  -e KAFKA_TRANSACTION_STATE_LOG_MIN_ISR=1 \
  -e KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS=0 \
  apache/kafka:4.2.2
```

Wait for Kafka to finish starting before launching Schema Registry.

## Step 0 continued: create the chat topic

Create a topic for the event stream (skip creation if it already exists):

```bash
docker exec kafkaServer /opt/kafka/bin/kafka-topics.sh \
  --create --topic group-chat-events \
  --bootstrap-server kafkaServer:9092 \
  --partitions 1 --replication-factor 1

docker exec kafkaServer /opt/kafka/bin/kafka-topics.sh \
  --describe --topic group-chat-events \
  --bootstrap-server kafkaServer:9092
```

This topic gives our event contract a concrete context. The Registry API steps below register schema metadata; they do not publish messages to this topic.

## Step 0 continued: launch Schema Registry

Use a Confluent Platform 8.2.x image with Kafka 4.2.x. The Registry connects to Kafka on the shared Docker network and stores its schema history in its own internal Kafka topic. Replication factor `1` is for this single-broker classroom setup only.

```bash
docker run --rm -d --name schemaRegistry --network tap -p 8081:8081 \
  -e SCHEMA_REGISTRY_HOST_NAME=schemaRegistry \
  -e SCHEMA_REGISTRY_LISTENERS=http://0.0.0.0:8081 \
  -e SCHEMA_REGISTRY_KAFKASTORE_BOOTSTRAP_SERVERS=PLAINTEXT://kafkaServer:9092 \
  -e SCHEMA_REGISTRY_KAFKASTORE_TOPIC_REPLICATION_FACTOR=1 \
  confluentinc/cp-schema-registry:8.2.2
```

Wait for the service, then verify its REST API from the host:

```bash
docker logs -f schemaRegistry
# Once it reports that it has started, press Ctrl+C to stop following logs.
curl -i http://localhost:8081/subjects
```

A fresh Registry should return an empty subject list (`[]`). Keep both containers running for the API walkthrough.

## Step 1: inspect the Registry

Start with the catalog of registered subjects:

```bash
curl -X GET http://localhost:8081/subjects
```

On a fresh registry the response is usually `[]`.

**Ask the class:** Is a subject a Kafka topic? (No: it is a Registry name for a schema's version history.)

## Step 2: register the first schema

Register an Avro value schema under the subject `student-questions-value`:

```bash
curl -X POST -H "Content-Type: application/vnd.schemaregistry.v1+json" --data '{"schema":"{\"type\":\"record\",\"name\":\"Question\",\"fields\":[{\"name\":\"text\",\"type\":\"string\"}]}"}' http://localhost:8081/subjects/student-questions-value/versions
```

The response contains an `id`, for example `{"id":1}`. The exact ID can differ. The body is a Registry API request: the outer JSON contains a `schema` string whose value is the Avro schema.

## Step 3: find the subject and its versions

List subjects again, then list the versions registered for our subject:

```bash
curl -X GET http://localhost:8081/subjects
curl -X GET http://localhost:8081/subjects/student-questions-value/versions
```

Expected shape of the responses:

```json
["student-questions-value"]
[1]
```

**Point out:** subject version `1` is not a Kafka offset and is not the schema ID.

## Step 4: retrieve a registered schema

Fetch the latest version stored under the subject:

```bash
curl -X GET http://localhost:8081/subjects/student-questions-value/versions/latest
```

The response includes the subject, version, schema ID, and schema. The `schema` value is JSON encoded inside the response JSON, so it appears escaped.

The Registry API can also fetch a schema by ID (`GET /schemas/ids/{id}`). A schema ID is a Registry identifier; it is not the subject version.

## Step 5: check a proposed evolution

Before registering a change, test a candidate schema that adds an optional `language` field with an Avro default:

```bash
curl -X POST -H "Content-Type: application/vnd.schemaregistry.v1+json" --data '{"schema":"{\"type\":\"record\",\"name\":\"Question\",\"fields\":[{\"name\":\"text\",\"type\":\"string\"},{\"name\":\"language\",\"type\":[\"null\",\"string\"],\"default\":null}]}"}' http://localhost:8081/compatibility/subjects/student-questions-value/versions/latest
```

With the default `BACKWARD` policy, the response should report `{"is_compatible":true}`. This checks compatibility against the latest schema; it does not register the candidate.

## Step 6: register the compatible version

Only after discussing the compatibility result, register the candidate under the same subject:

```bash
curl -X POST -H "Content-Type: application/vnd.schemaregistry.v1+json" --data '{"schema":"{\"type\":\"record\",\"name\":\"Question\",\"fields\":[{\"name\":\"text\",\"type\":\"string\"},{\"name\":\"language\",\"type\":[\"null\",\"string\"],\"default\":null}]}"}' http://localhost:8081/subjects/student-questions-value/versions
curl -X GET http://localhost:8081/subjects/student-questions-value/versions
```

The versions should now include `1` and `2`. Registration stores a schema version; it still does not write a record to Kafka.

## Step 7: inspect compatibility configuration

The quickstart shows how to query the global compatibility configuration:

```bash
curl -X GET http://localhost:8081/config
```

A fresh Confluent Schema Registry defaults to `BACKWARD`. Compatibility can also be configured per subject. In production, the setting is part of the data-contract decision: agree on it deliberately rather than changing it casually during a demo.

## Discuss: what if we rename `text`?

Before running another compatibility check, ask the class to predict the result if the proposed schema removes `text` and adds a required `question` field.

Then test that candidate against `student-questions-value` using the same compatibility endpoint.

**Expected discussion:** under `BACKWARD`, the new reader must be able to read old records. Old records have `text`, not `question`; without a suitable default/alias strategy, the proposed schema is not backward compatible.

The check is a structural guardrail, not a substitute for reviewing field meaning and consumer behavior.

## What did the API exercise demonstrate?

We used the Registry directly to:

1. Register a schema under a subject.
2. List subjects and schema versions.
3. Retrieve the latest schema.
4. Check a proposed schema for compatibility before registering it.
5. Inspect the compatibility policy.

This was a Registry exercise, not a Kafka producer/consumer exercise: no message was sent through Kafka. Next lessons will show how client-side serializers, Connect converters, and Streams SerDes can make these API interactions part of reading and writing records.

## Guided class discussion

A team wants to add `language` to the question event. As a class, decide:

1. What should the field mean, and can it be absent?
2. What change would you propose to the schema?
3. Which direction of compatibility matters if new code must read old records?
4. What should be checked before the producer starts publishing the new version?

**Suggested conclusion:** define the field's meaning first, make it optional with a format-appropriate default, check the proposed version against the subject's policy, and test old and new records with the readers that will be deployed.

## Where we go next

Today we have treated schemas as contracts around Kafka records. Next, we can see how other Kafka ecosystem tools use those contracts:

- Kafka Connect can use registry-aware converters when moving records.
- Kafka Streams applications can use registry-aware serializers and deserializers.

Those tools are separate topics; understanding Schema Registry does not require us to assume their configuration or APIs.

## Recap

- Kafka carries records; Schema Registry manages schemas and their versions.
- Confluent Schema Registry is a Confluent project, separate from Apache Kafka.
- A schema format, a schema, a subject, and a schema ID are different concepts.
- Compatibility helps us evolve records safely, but does not guarantee that business meaning is unchanged.

**Final question:** What would you check before changing the event that your producer publishes?

## References

- [Confluent Schema Registry source repository](https://github.com/confluentinc/schema-registry)
- [Confluent Schema Registry documentation](https://docs.confluent.io/platform/current/schema-registry/index.html)
- [Schema evolution and compatibility](https://docs.confluent.io/platform/current/schema-registry/fundamentals/schema-evolution.html)
- [Schema Registry API usage](https://docs.confluent.io/platform/current/schema-registry/develop/api.html)
- [Kafka documentation](https://kafka.apache.org/documentation/)